In [1]:
import sys
from pathlib import Path

sys.path.append(f"{Path().absolute().parent}")

In [2]:
from __future__ import annotations

from MRO_library import *
import plots
from radp.digital_twin.utils.gis_tools import GISTools

### Data Preprocessing

In [3]:
ue_data = pd.read_csv('data/sim_data/UE_Data_500UE_500ticks_10batch.csv')
# ue_data = pd.read_csv("data/sim_data/100UE_500Ticks.csv")
# ue_data = ue_data.rename(
#    columns={"lon": "longitude", "lat": "latitude"}
#)  # Delete Later

ue_data

,mock_ue_id,longitude,latitude,tick
0,0,-157.649256,21.547852,0
1,1,-15.004199,77.913292,0
2,2,-133.578957,86.550651,0
3,3,-125.162643,-77.464666,0
4,4,47.623154,-65.160734,0
...,...,...,...,...
2499995,495,-140.051518,-71.407540,499
2499996,496,114.691440,-89.545954,499
2499997,497,-0.718062,25.229923,499
2499998,498,-82.468220,20.874951,499


In [4]:
topology = pd.read_csv("data/sim_data/topology.csv")

topology.loc[topology["cell_id"] == "cell_1", "cell_lat"] = -90
topology.loc[topology["cell_id"] == "cell_2", "cell_lat"] = 0
topology.loc[topology["cell_id"] == "cell_3", "cell_lat"] = 90

topology.loc[topology["cell_id"] == "cell_1", "cell_lon"] = -180
topology.loc[topology["cell_id"] == "cell_2", "cell_lon"] = 0
topology.loc[topology["cell_id"] == "cell_3", "cell_lon"] = 180

topology.loc[topology["cell_id"] == "cell_1", "cell_carrier_freq_mhz"] = 2100
topology.loc[topology["cell_id"] == "cell_2", "cell_carrier_freq_mhz"] = 2100
topology.loc[topology["cell_id"] == "cell_3", "cell_carrier_freq_mhz"] = 2100

topology

,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz
0,-90.0,-180.0,cell_1,0,2100
1,0.0,0.0,cell_2,120,2100
2,90.0,180.0,cell_3,240,2100


In [5]:
data = concatenate_ue_to_topology(ue_data, topology)
data = connect_ue_to_all_cells(data, topology)

In [ ]:
# Calculating distance and received power
data["distance_km"] = data.apply(
    lambda row: GISTools.get_log_distance(
        row["latitude"], row["longitude"], row["cell_lat"], row["cell_lon"]
    ),
    axis=1,
)
data["cell_rxpower_dbm"] = data.apply(
    lambda row: calculate_received_power(
        row["distance_km"], row["cell_carrier_freq_mhz"]
    ),
    axis=1,
)
# Rename Mock UE ID to UE ID for consistency
data = data.rename(columns={"mock_ue_id": "ue_id"})
data["ue_id"] = data["ue_id"].astype(int)
topology["cell_id"] = topology["cell_id"].str.replace("cell_", "").astype(int)
data["cell_id"] = data["cell_id"].str.extract("(\d+)").astype(int)
data = data.rename(columns={"latitude": "loc_y", "longitude": "loc_x"})
data["relative_bearing"] = data.apply(
    lambda row: GISTools.get_relative_bearing(
        row["cell_az_deg"], row["cell_lat"], row["cell_lon"], row["loc_y"], row["loc_y"]
    ),
    axis=1,
)
# data = data.rename(columns={'cell_rxpower_dbm': 'rsrp_dbm'})
data = add_sinr_column(data)

### Perform Attachment w/ Hyst & TTT

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(data["sinr_db"], bins=50, kde=True, color='skyblue', edgecolor='black')
plt.title("Distribution of SINR (dB)")
plt.xlabel("SINR (dB)")
plt.ylabel("Count")
plt.grid(True)
plt.tight_layout()
plt.show()

#### HyperParameters

In [ ]:
hyst = 0.25  # 0<hyst
ttt = 5  # 2<ttt< total ticks
rlf_threshold = -25.25

In [ ]:
attached_df = perform_attachment_hyst_ttt(data, hyst, ttt, rlf_threshold)

In [ ]:
attached_df

In [ ]:
mro_metric = calculate_mro_metric(attached_df)
print(f"Switches: {count_switches(attached_df)}, RLF: {count_rlf(attached_df)}")
print(f"MRO Score: {mro_metric}")

In [ ]:
plots.plot_scatter2(attached_df, topology)

In [ ]:
# ue_ids = attached_df['ue_id'].unique()
# for i in ue_ids:
#     plots.plot_sinr_over_time(attached_df, data, i, rlf_threshold)
#     plots.individual_scatter_plot(attached_df, data, i)

## Optimizing Hyst & TTT

In [ ]:
max_diff = find_hyst_diff(attached_df)
num_ticks = attached_df["tick"].nunique()
print(f"Max Hyst Diff: {max_diff}\nNum Ticks: {num_ticks}")

In [ ]:
epochs = 10

hyst_range = [0, max_diff]
ttt_range = [2, num_ticks+1]
score = pd.DataFrame(columns=["hyst", "ttt", "score"])

### Random Sampling

In [ ]:
# Define the header for the columns
header = f"{'Epoch':<6} {'Hyst':<14} {'TTT':<6} {'MRO Metric':<12}"
print(header)
print("-" * len(header))  # Automatically match the length of the header

score.loc[len(score)] = [hyst, ttt, mro_metric]
# Loop over the epochs
for i in range(epochs):
    while True:
        hyst = np.random.uniform(hyst_range[0], hyst_range[1])
        ttt = np.random.randint(ttt_range[0], ttt_range[1])
        if ttt not in score["ttt"].values or hyst not in score["hyst"].values:
            break

    # Perform attachment and calculate MRO Metric
    attached_df = perform_attachment_hyst_ttt(data, hyst, ttt, rlf_threshold)
    mro_metric = calculate_mro_metric(attached_df)

    # Print the data, ensuring proper alignment with the header
    print(f"{i:<6} {hyst:<14.10f} {ttt:<6} {mro_metric:<12.6f}")

    # Store the data in the score DataFrame
    score.loc[len(score)] = [hyst, ttt, mro_metric]


In [ ]:
score.loc[score["score"].idxmax()]

In [ ]:
plots.plot_3d_hyst_ttt_score(score)

### Baysian Optimization

In [ ]:
from skopt import gp_minimize

# Define the objective function for Bayesian Optimization
def objective(params):
    hyst, ttt = params
    # Perform attachment with the current hyst, ttt
    attached_df = perform_attachment_hyst_ttt(data, hyst, int(ttt), rlf_threshold)
    # Calculate the MRO metric
    mro_metric = calculate_mro_metric(attached_df)
    # Return the negative MRO metric because we want to maximize it

    print(f"Testing Hyst: {hyst:.10f}, TTT: {ttt}, MRO Metric: {mro_metric:.6f}")
    score.loc[len(score)] = [hyst, ttt, mro_metric]

    return -mro_metric

In [ ]:
# Define the search space for hyst and ttt
hyst_range = [0, max_diff]
ttt_range = [2, num_ticks + 1]

# Define the bounds for the optimizer
space = [
    (hyst_range[0], hyst_range[1]),  # Bound for hyst
    (ttt_range[0], ttt_range[1])     # Bound for ttt
]

score = pd.DataFrame(columns=["hyst", "ttt", "score"])

In [ ]:
# Run Bayesian Optimization
result = gp_minimize(objective,                  # The objective function
                     space,                      # The search space
                     n_calls=epochs,             # Number of iterations
                     random_state=9,            # For reproducibility
                     verbose=False)               # Print optimization progress

# After optimization, print the best found parameters
best_hyst, best_ttt = result.x
best_mro_metric = -result.fun  # Negate back to get the maximum MRO metric

print(f"Optimized Hyst: {best_hyst:.10f}, Optimized TTT: {best_ttt}, MRO Metric: {best_mro_metric:.6f}")

In [ ]:
plots.plot_3d_hyst_ttt_score(score)